<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Select useful features

**[Select useful features](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/)** · Machine Learning: From Problem to Reliable Model · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** test new features with cross-validation on train, tell useful, redundant and leaking features apart, recognise a leak because it looks too good, write the decisions in the experiment log, and add the chosen feature to `ticket_model.py`.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Cross-validation and its spread, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#cross-validation). The experiment log, from [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log). Leakage and target timing, from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/). |
| **You do not need** | The local project. The setup below downloads the data and writes `ticket_model.py` as it was after Module 4. You change it in this notebook. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M06-L01-select-useful-features/select-useful-features-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 14 rows that you added in the earlier lessons (E01 to E14), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 14 rows, from E01 to E14.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
    ('E06', 'Decision tree, max_depth=3', 0.234, 0.011, None, 'reject', 'Far below E03.'),
    ('E07', 'Decision tree, max_depth=6', 0.258, 0.010, None, 'reject', 'The best of the three trees, still far below E03.'),
    ('E08', 'Decision tree, no depth limit', 0.137, 0.004, None, 'reject', 'It memorises train (train AP 1.000) and fails on new tickets.'),
    ('E09', 'Decision tree, no depth limit, min_samples_leaf=100', 0.278, 0.017, None, 'reject', 'The best tree so far, still far below E03.'),
    ('E10', 'E03 with C=0.0001 (a very strong penalty)', 0.302, 0.012, None, 'reject', 'It underfits: both train and CV AP fall.'),
    ('E11', 'Random forest, 200 trees, default settings', 0.290, 0.014, None, 'reject', 'Its trees memorise (train AP 1.000); below E03.'),
    ('E12', 'Random forest, 200 trees, min_samples_leaf=20', 0.325, 0.018, 0.382, 'reject', 'A tie with E03 (higher on 3 folds of 5), and 2,000 times bigger as a file.'),
    ('E13', 'Gradient boosting, default settings', 0.316, 0.010, 0.369, 'reject', 'Lower than E03 on all 5 folds.'),
    ('E14', 'Gradient boosting, max_depth=3, learning_rate=0.05', 0.327, 0.016, 0.374, 'reject', 'A tie with E03; more complex and harder to explain.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three questions for every candidate

A student who reads the answer key before each practice exercise gets every exercise right, and learns nothing. A column that secretly contains the answer does the same to a model. Mei offers four new columns. Each one is a **candidate feature**: a column that could become a feature, if it passes three questions.

1. **Is it known when the ticket is created?** If not, it leaks.
2. **Does it add information that the model does not have yet?** If not, it is redundant.
3. **Does CV on train say it helps, on more than the spread?** If not, leave it out: a simpler model is better.

So each candidate is one of three kinds: **useful** if it passes all three questions, **redundant** if it repeats another feature or adds less than the spread, and **leaking** if it is written after the prediction time. Only a useful candidate is kept; the experiment log records the others as `reject`, with the reason. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#eight-tickets-by-hand) works through the questions on 8 tickets by hand, and shows when each column is written on a time line.

Run the next cell. It loads train and valid with the 10 features, and makes the course's 5 folds.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ticket_model import CATEGORIES, NUMBERS, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print(len(CATEGORIES) + len(NUMBERS), "features;", len(train), "train and", len(valid), "valid tickets")

> **Check.** You should see `10 features; 16939 train and 1179 valid tickets`.

## 2. The four candidates

`extra_features.csv` has the candidates, one row per ticket. Run the cell. It joins them to train and valid by `ticket_id`.

In [ ]:
extra = pd.read_csv("data/extra_features.csv")
candidates = train.merge(extra, on="ticket_id", how="left", validate="one_to_one")
candidates_valid = valid.merge(extra, on="ticket_id", how="left", validate="one_to_one")
print(len(extra), "rows in extra_features.csv; train after the merge:", len(candidates))
print(candidates[["ticket_id", "prior_escalations_90d", "order_value_band",
                  "customer_escalation_rate", "priority_now", TARGET]].head(4).to_string(index=False))

> **Check.** You should see `19120 rows in extra_features.csv; train after the merge: 16939`, and four rows.

**`prior_escalations_90d`**: the customer's escalations in the 90 days before the ticket. Does more history mean more risk? Run the cell (5 means "5 or more").

In [ ]:
history = candidates["prior_escalations_90d"].clip(upper=5)  # 5 means "5 or more"
print(candidates.groupby(history)[TARGET].agg(["size", "mean"]).round(3))

> **Check.** 0 earlier escalations: 9,130 tickets, 0.091 escalated. 1: 0.119. 2: 0.144. 5 or more: 1,083 tickets, 0.252.

The rate rises with the history, and the value is known at creation: it counts only earlier tickets.

**`order_value_band`**: `none`, `low`, `mid` or `high`. Run the cell to see where the bands come from.

In [ ]:
print(candidates.groupby("order_value_band")["order_value"].agg(["size", "min", "max"]))
print("tickets with no order_value:", candidates["order_value"].isna().sum())

> **Check.** `low` goes from 4.50 to 49.98, `mid` from 50.00 to 199.98, `high` from 200.05. `none` has 2,462 tickets, the same as the tickets with no `order_value`.

The band is `order_value` cut into four pieces. It repeats a feature that the model already has.

**`priority_now`**: the priority at the time of the export. Run the cell.

In [ ]:
print(pd.crosstab(candidates["priority_now"], candidates[TARGET]))

> **Check.** Every escalated ticket (1,983) has `priority_now` 1. No ticket with 2 or 3 escalated.

The help desk sets the priority to 1 when it escalates a ticket. The column does not predict the escalation: it records it.

**`customer_escalation_rate`**: the customer's escalation rate. Over which tickets? Run the cell. It checks one customer, C-00078.

In [ ]:
everything = pd.concat([load("train"), load("valid"), load("test")])
customer = "C-00078"
print("customer_escalation_rate in the file:",
      candidates.loc[candidates["customer_id"] == customer, "customer_escalation_rate"].iloc[0])
print("rate over all labelled tickets:      ", round(everything.loc[everything["customer_id"] == customer, TARGET].mean(), 4))
print("rate over the train tickets only:    ", round(train.loc[train["customer_id"] == customer, TARGET].mean(), 4))

> **Check.** You should see `0.0959` twice, then `0.0769`.

The value in the file is the rate over **all** labelled tickets, including May and June, and including each ticket's own target. On 1 May, Larkfield knew only the train tickets: 0.0769. The column contains the future.

## 3. Worked example: decide with CV on train

The next cell defines two helpers. `pipeline_for()` builds the same steps as `build_pipeline()`, for other lists of columns. `cv_folds()` returns the AP on each of the 5 folds. Run it. It prints nothing.

In [ ]:
def pipeline_for(categories, numbers):
    """The steps of build_pipeline(), for other lists of columns."""
    number_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                             ("scale", StandardScaler())])
    prepare = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), categories),
                                 ("numbers", number_steps, numbers)])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])


def cv_folds(data, categories, numbers):
    """The AP on each of the 5 folds of train, for logistic regression with these columns."""
    return cross_val_score(pipeline_for(categories, numbers), data[categories + numbers], data[TARGET],
                           cv=cv, scoring="average_precision")

Now test each candidate: the 10 features plus one column. All five models use the **same folds**, so you can compare them fold by fold.

> **Predict.** Which candidate will add the most? Which result will look "too good to be true"? Run the cell. It takes about 10 seconds.

In [ ]:
tests = {
    "(nothing)": (CATEGORIES, NUMBERS),
    "prior_escalations_90d": (CATEGORIES, NUMBERS + ["prior_escalations_90d"]),
    "order_value_band": (CATEGORIES + ["order_value_band"], NUMBERS),
    "customer_escalation_rate": (CATEGORIES, NUMBERS + ["customer_escalation_rate"]),
    "priority_now": (CATEGORIES, NUMBERS + ["priority_now"]),
}
folds = {added: cv_folds(candidates, *columns) for added, columns in tests.items()}
for added, fold_scores in folds.items():
    gain = fold_scores - folds["(nothing)"]
    print(f"+ {added:25s} CV AP {fold_scores.mean():.3f} ± {fold_scores.std():.3f}   gain per fold {gain.round(3)}")

> **Check.** You should see:
>
> | Added | CV AP |
> |---|---|
> | (nothing) | 0.327 ± 0.012 |
> | `prior_escalations_90d` | 0.339 ± 0.016, gain per fold `[0.01  0.01  0.005 0.025 0.013]` |
> | `order_value_band` | 0.328 ± 0.014, gain per fold `[ 0.007 -0.005  0.004  0.001  0.001]` |
> | `customer_escalation_rate` | 0.445 ± 0.015 |
> | `priority_now` | 0.851 ± 0.024 |

Some folds are harder for every model, so much of the spread is the same for both models. When two models use the same folds, look at the **gain on each fold**. `prior_escalations_90d` helps on all 5 folds. `order_value_band` helps on some and hurts on one: no real difference.

Two candidates jump far above the rest. One extra column takes the AP from 0.327 to 0.851. Honest features moved it by about 0.01. **"Too good to be true" is the signal for a leak.** Then question 1 decides: both columns are written after the ticket is created.

> **Your turn.** Write the kind of each candidate: `"useful"`, `"redundant"` (it repeats a feature, or its gain is smaller than the spread) or `"leaking"`. Run the cell, then the check cell. Only the useful candidate is kept; section 5 writes all four in the log, with `keep` or `reject`.

In [ ]:
# Your turn: for each candidate, write its kind: "useful", "redundant" or "leaking".
kinds = {
    "prior_escalations_90d": "",
    "order_value_band": "",
    "customer_escalation_rate": "",
    "priority_now": "",
}

In [ ]:
if any(v not in ("useful", "redundant", "leaking") for v in kinds.values()):
    print('Not yet: write "useful", "redundant" or "leaking" for every candidate.')
else:
    expect_hash("your set of kinds", kinds, "27dfcc3a8de3834b")

## 4. Confirm on the validation month

You made the decision with train only. Now confirm it once on May. The order matters: if you choose by the validation score, May becomes part of the choice and stops being an independent check. And May is small (183 escalations): its AP moves by about 0.03 by chance.

Run the cell.

In [ ]:
for added, (categories, numbers) in tests.items():
    pipeline = pipeline_for(categories, numbers).fit(candidates[categories + numbers], candidates[TARGET])
    scores = pipeline.predict_proba(candidates_valid[categories + numbers])[:, 1]
    print(f"+ {added:25s} valid AP {average_precision_score(valid[TARGET], scores):.3f}"
          f"  ROC AUC {roc_auc_score(valid[TARGET], scores):.3f}")

> **Check.** (nothing): `0.353` and `0.738`. `prior_escalations_90d`: `0.391` and `0.752`. `order_value_band`: `0.358` and `0.738`. `customer_escalation_rate`: `0.485` and `0.812`. `priority_now`: `0.873` and `0.981`.

May agrees with the decision. But look at the two leaks: they look great on May too. The leaked information is in the May rows as well, so the validation month cannot catch a leak. Only the question "when is the value written?" catches it.

## 5. Write it in the experiment log

In Module 5 you started an experiment log, `experiment_log.csv`: one row per experiment, with its CV score, its valid score, the decision and the reason. It has the rows E01 to E14. A rejected idea belongs in the log too, so nobody tries `priority_now` again next month.

The next cell adds the four candidates as E15 to E18, with the fold scores from section 3. Run it.

In [ ]:
for row_id, added, valid_ap, decision, reason in [
    ("E15", "prior_escalations_90d", 0.391, "keep", "Known at creation, and higher than E03 on all 5 folds."),
    ("E16", "order_value_band", 0.358, "reject", "It repeats order_value; the gain is smaller than the noise."),
    ("E17", "customer_escalation_rate", 0.485, "reject", "Leak: computed over later targets, so not known at creation."),
    ("E18", "priority_now", 0.873, "reject", "Leak: the help desk sets it to 1 when a ticket escalates."),
]:
    change = f"E03 + {added}" + (" (11 features)" if row_id == "E15" else "")
    log_experiment(row_id, change, folds[added].mean(), folds[added].std(), valid_ap, decision, reason)
show_log()

> **Check.** You should see 18 rows. E15 (`0.339`, `0.016`, `0.391`) is `keep`; E16 (redundant), E17 and E18 (leaking) are `reject`, and each reason says why.

E15 replaces E03 as the model to keep. Lesson 2 adds the tuning experiments.

## 6. Add the feature to ticket_model.py

The decision: add `prior_escalations_90d`. Two changes in `ticket_model.py`:

1. `NUMBERS` gets a seventh entry, `"prior_escalations_90d"`.
2. `load()` reads the part, then merges `prior_escalations_90d` from `data/extra_features.csv` on `ticket_id`, with `how="left"` (keep every ticket) and `validate="one_to_one"` (stop if a ticket appears twice).

> **Your turn.** Make the two changes in the next cell, at the two `Your turn` marks. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#change-ticket_modelpy) shows the changes as a diff. Run the cell, then the check cell.

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
# Your turn (1 of 2): add "prior_escalations_90d" at the end of NUMBERS.
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    # Your turn (2 of 2): read the part, then merge prior_escalations_90d from
    # data/extra_features.csv on ticket_id (how="left", validate="one_to_one").
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

The check cell reloads `ticket_model` (Python keeps the old version until you reload it), loads train and valid again, and checks the new file.

In [ ]:
import importlib

import ticket_model

importlib.reload(ticket_model)  # read the new file
from ticket_model import CATEGORIES, FEATURES, NUMBERS, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
if "prior_escalations_90d" not in NUMBERS:
    print("Not yet: NUMBERS has", len(NUMBERS), "entries. Add prior_escalations_90d at the end.")
elif "prior_escalations_90d" not in valid.columns:
    print("Not yet: load() does not add prior_escalations_90d. Merge extra_features.csv in load().")
elif len(valid) != 1179 or valid["prior_escalations_90d"].isna().any():
    print('Not yet: the merge changed the rows or left gaps. Use how="left" and validate="one_to_one".')
else:
    expect("the number of features", len(FEATURES), 11)

> **Check.** You should see `Check passed: the number of features is 11.`

Run the next cell: the final model of the course, fitted on train, scored on May.

In [ ]:
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
print(len(FEATURES), "features:", FEATURES)
print("valid AP", round(average_precision_score(valid[TARGET], scores), 3),
      " ROC AUC", round(roc_auc_score(valid[TARGET], scores), 3))

> **Check.** You should see 11 features, ending with `prior_escalations_90d`, and `valid AP 0.391  ROC AUC 0.752`.

From now on, every lesson uses this file.

## 7. Your turn: remove a feature instead

Change one condition: remove a feature instead of adding one. Test `region` and `channel`, each removed from the 11 features, on the same folds.

> **Predict.** Will the model lose anything without `region`? Without `channel`?

> **Your turn.** Replace `...` with a call to `cv_folds(train, ...)` with both lists without `removed`. Run the cell.

In [ ]:
# Your turn: remove one feature at a time, with the same folds.
all_folds = cv_folds(train, CATEGORIES, NUMBERS)
print(f"all {len(FEATURES)} features      CV AP {all_folds.mean():.3f} ± {all_folds.std():.3f}")
for removed in ["region", "channel"]:
    fold_scores = ...  # cv_folds(train, ...) with CATEGORIES and NUMBERS without `removed`
    if fold_scores is not ...:
        change = fold_scores - all_folds
        print(f"without {removed:10s} CV AP {fold_scores.mean():.3f} ± {fold_scores.std():.3f}   change per fold {change.round(3)}")

In [ ]:
if fold_scores is ...:
    print("Not yet: replace ... with cv_folds(train, ...), then run the cell again.")
else:
    expect_hash('the CV AP without channel', round(float(fold_scores.mean()), 3), 'd85efb6485efddcf')

Compare your reading with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#your-turn-remove-a-feature-instead).

## 8. Failure case: the leak in production

Tomás reports the model with `priority_now`: valid AP 0.873. What would it do with real new tickets? When a ticket is created, `priority_now` is simply the priority. Run the cell.

In [ ]:
categories, numbers = tests["priority_now"]  # the 10 features + priority_now
columns = categories + numbers
leaky = pipeline_for(categories, numbers).fit(candidates[columns], candidates[TARGET])
reported = leaky.predict_proba(candidates_valid[columns])[:, 1]
at_creation = candidates_valid.assign(priority_now=candidates_valid["priority"])  # what a new ticket has
real = leaky.predict_proba(at_creation[columns])[:, 1]
print("valid AP as reported:               ", round(average_precision_score(valid[TARGET], reported), 3))
print("valid AP with the value at creation:", round(average_precision_score(valid[TARGET], real), 3))

> **Check.** You should see `0.873`, then `0.353`.

The report says 0.873. With the values a new ticket really has, the same model gets 0.353, no better than the model without the column. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#the-leak-that-validation-could-not-catch) has the full failure case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you tested four candidates with CV on train, fold by fold, and kept one: `prior_escalations_90d`. You rejected a redundant column and two leaking ones, which you caught because they looked too good, and you wrote each decision in the experiment log. `ticket_model.py` now has 11 features: valid AP 0.391, ROC AUC 0.752. Lesson 2 asks if other settings of the model would do better.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.